# Instance creation stats




## 1. Environment setup


In [25]:
# !pip install pandas requests

import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [26]:

%run folio_auth.ipynb


Login succeeded. Token retrieved.


## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [27]:
def fetch_all_records(endpoint, records_key, limit=1000):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = OKAPI_URL.copy()
    headers = HEADERS.copy()


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records


## 4. Get # of Days to Search from Input
The number entered will be used to search for Instance records with a date created of {number of days} - today's date. You can choose to bypass this and use a standard calculation if you prefer not to have an input.


In [28]:

while True:
    try:
        from_days = int(input("Enter the number of days to search back from today, or press Enter to use the default of 90 days: ") or 90)
        if from_days < 0:  
            print("Number cannot be negative")
            continue
        else:                
            date_x_days_ago = datetime.now() - timedelta(days=from_days)
            search_date =str(date_x_days_ago.strftime("%Y-%m-%d"))
            print("You entered:", from_days, "days. The search will look for Instance records created since: ", search_date)
            break
    except ValueError:
        print("Please enter a valid number.")



You entered: 90 days. The search will look for Instance records created since:  2026-05-15


## 4. Pull data from each endpoint

TODO (FOLIO-specific): confirm the `records_key` for each — FOLIO's convention is
usually the plural of the resource, but it varies (e.g. `/groups` often returns
`"usergroups"` rather than `"groups"` — **check this**, I'm not certain of the exact
key for your instance).


In [29]:

def fetch_all_records(endpoint, records_key, limit=100, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records


staff_raw = fetch_all_records(
    "/users",
    records_key="users",
    query='type=="staff"',
) 


# instances_raw = fetch_all_records(
#     "/inventory/instances",
#     records_key="instances",
#     query='metadata.createdDate >= ' + search_date,
# )

instances_raw = fetch_all_records(
    "/instance-storage/instances",
    records_key="instances",
    query='metadata.createdDate >= ' + search_date,
)

print(f"staff:     {len(staff_raw)}")
print(f"instances: {len(instances_raw)}")


staff:     231
instances: 108


In [30]:
# Inspect staff records
staff_df    = pd.DataFrame(staff_raw)
staff_df.head()

,username,id,barcode,active,type,patronGroup,departments,proxyFor,personal,expirationDate,createdDate,updatedDate,metadata,customFields,preferredEmailCommunication,externalSystemId,tags,enrollmentDate
0,scurry,a8046409-430a-41f0-87a0-9c7e0a19f9ee,90042,False,staff,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,[],[],"{'lastName': 'Curry', 'firstName': 'Stephen', ...",2026-07-01T03:59:59.999+00:00,2026-05-20T15:44:46.180+00:00,2026-05-20T15:44:46.180+00:00,{'createdDate': '2024-04-17T14:54:56.165+00:00...,"{'activePatronGroups': [], 'genero': 'opt_0'}",[],NaN,NaN,NaN
1,postman,1bc06b9d-0e6e-4ac2-91e1-1fb6677ba6f5,postman,False,staff,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,[],[],"{'lastName': 'User', 'firstName': 'Postman', '...",2026-01-01T04:59:59.000+00:00,2024-12-13T17:59:31.261+00:00,2024-12-13T17:59:31.261+00:00,{'createdDate': '2024-12-13T17:58:50.222+00:00...,{'activePatronGroups': []},[],NaN,NaN,NaN
2,EBSCOGCSarainio,c59023c4-7838-4e67-a028-9c143123dda4,NaN,False,staff,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,[],[],"{'lastName': 'EBSCOGCSarainio', 'firstName': '...",2025-08-12T16:39:13.000+00:00,2025-08-12T14:39:44.573+00:00,2025-08-12T14:39:44.573+00:00,{'createdDate': '2025-05-29T12:00:04.685+00:00...,NaN,[],NaN,NaN,NaN
3,chutchinson,5e20afa6-13e3-484b-b861-9956204c9ba4,NaN,True,staff,749e4843-fe70-403b-b428-e8d471432377,[],[],"{'lastName': 'Hutchinson', 'firstName': 'Corri...",NaN,2026-01-21T17:30:07.727+00:00,2026-01-21T17:30:07.727+00:00,{'createdDate': '2025-12-30T00:21:18.792+00:00...,{'activePatronGroups': []},[],NaN,NaN,NaN
4,ibrown,c1c0f905-e01f-49d3-9eb4-d8ae44b9416c,22874000505634,False,staff,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,[],[],"{'lastName': 'Brown', 'firstName': 'Isabella',...",2024-12-01T04:59:59.000+00:00,2025-03-07T01:28:14.407+00:00,2025-03-07T01:28:14.407+00:00,{'createdDate': '2022-04-12T22:36:55.676+00:00...,"{'activePatronGroups': [], 'campusAffiliation'...",[],ibrown,NaN,NaN


In [31]:
#Inspect instance records
instances_df   = pd.DataFrame(instances_raw)
instances_df.head()


,id,_version,hrid,source,title,indexTitle,alternativeTitles,editions,series,identifiers,contributors,subjects,classifications,publication,publicationFrequency,publicationRange,electronicAccess,dates,instanceTypeId,instanceFormatIds,instanceFormats,physicalDescriptions,languages,notes,administrativeNotes,modeOfIssuanceId,catalogedDate,previouslyHeld,staffSuppress,discoverySuppress,deleted,statisticalCodeIds,statusId,statusUpdatedDate,tags,metadata,holdingsRecords2,natureOfContentTermIds
0,fff6cbc4-89d8-4b11-912e-899988eef596,1,in00000425095,MARC,12 things to know about political parties / Vi...,12 things to know about political parties /,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[],"[{'value': '23766617', 'identifierTypeId': '7e...","[{'name': 'Hayes, Vicki C.', 'contributorTypeI...",[{'value': 'Political parties--United States--...,"[{'classificationNumber': 'JK2261 .H337 2025',...","[{'publisher': 'Black Rabbit Books', 'place': ...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[48 pages : illustrations ; 24 cm.],[eng],[{'instanceNoteTypeId': '86b6e817-e1bc-42fb-ba...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-20,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-20T18:12:50.827+0000,{'tagList': []},{'createdDate': '2026-07-20T18:12:50.826+00:00...,[],[]
1,4852f533-06d9-413e-b7c3-f97586631a4f,1,in00000425128,MARC,Writing the U.S. Constitution : framework of a...,Writing the U.S. Constitution : framework of a...,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[{'value': 'Perspectives library.'}],"[{'value': 'in00024787139', 'identifierTypeId'...","[{'name': 'Baxter, Roberta, 1952-', 'contribut...",[{'value': 'United States. Constitution--Juven...,"[{'classificationNumber': '342.73029', 'classi...","[{'publisher': 'Cherry Lake Press', 'place': '...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[32 pages: illustrations (some color) ; 24 cm.],[eng],[{'instanceNoteTypeId': '6a2533a7-4de2-4e64-84...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-21,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-21T13:51:44.819+0000,{'tagList': []},{'createdDate': '2026-07-21T13:51:44.818+00:00...,[],[]
2,d7b6c2a9-95ed-4856-91a2-647c8298e97a,1,in00000425230,MARC,Bicycling the Colorado Rockies / Vici De Haan.,Bicycling the Colorado Rockies /,[],"[Rev. & expanded, 2nd rd.]",[],"[{'value': '4749869', 'identifierTypeId': '7e5...","[{'name': 'DeHaan, Vici', 'contributorNameType...",[{'value': 'Bicycle touring--Rocky Mountains--...,[{'classificationNumber': 'GV1045.5.R6 D4 1985...,"[{'publisher': 'Pruett Pub. Co.', 'place': 'Bo...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[114 p. : ill. ; 22 cm.],[eng],[{'instanceNoteTypeId': '86b6e817-e1bc-42fb-ba...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,NaN,False,False,False,False,[],NaN,2026-07-22T20:15:00.786+0000,{'tagList': []},{'createdDate': '2026-07-22T20:15:00.785+00:00...,[],[]
3,2f756088-c4ad-41ff-8c0b-3936d9d9d8ed,2,in00000425296,MARC,Tōkyō Kokuritsu Hakubutsukan zuhan mokuroku....,Tōkyō Kokuritsu Hakubutsukan zuhan mokuroku....,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[],"[{'value': '5036355', 'identifierTypeId': '7e5...","[{'name': 'Tōkyō Kokuritsu Hakubutsukan', 'c...",[{'value': 'Tōkyō Kokuritsu Hakubutsukan--Ca...,"[{'classificationNumber': 'ND1055 .T627 1984',...",[{'publisher': 'Tōkyō Kokuritsu Hakubutsukan...,[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],"[x, 116 p. : chiefly ill. ; 27 cm.]","[jpn, Jpn, eng]",[],[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,NaN,False,False,False,False,[],NaN,2026-07-24T23:51:56.138+0000,{'tagList': []},{'createdDate'

## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [32]:
print(staff_df.columns.tolist())
print(instances_df.columns.tolist())



['username', 'id', 'barcode', 'active', 'type', 'patronGroup', 'departments', 'proxyFor', 'personal', 'expirationDate', 'createdDate', 'updatedDate', 'metadata', 'customFields', 'preferredEmailCommunication', 'externalSystemId', 'tags', 'enrollmentDate']
['id', '_version', 'hrid', 'source', 'title', 'indexTitle', 'alternativeTitles', 'editions', 'series', 'identifiers', 'contributors', 'subjects', 'classifications', 'publication', 'publicationFrequency', 'publicationRange', 'electronicAccess', 'dates', 'instanceTypeId', 'instanceFormatIds', 'instanceFormats', 'physicalDescriptions', 'languages', 'notes', 'administrativeNotes', 'modeOfIssuanceId', 'catalogedDate', 'previouslyHeld', 'staffSuppress', 'discoverySuppress', 'deleted', 'statisticalCodeIds', 'statusId', 'statusUpdatedDate', 'tags', 'metadata', 'holdingsRecords2', 'natureOfContentTermIds']


In [17]:
# Spot-check types of the columns you intend to join on
# Since the user ID in the instance records is nested in the metadata dictionary, we need to extract it first
print(staff_df['id'].dtype)
print(instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None).dtype)

str
str


## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [33]:
instances_users = instances_df.merge(staff_df, left_on=instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None), right_on='id', how='left', suffixes=('_instance', '_user')
)

instances_users.head()


,id,id_instance,_version,hrid,source,title,indexTitle,alternativeTitles,editions,series,identifiers,contributors,subjects,classifications,publication,publicationFrequency,publicationRange,electronicAccess,dates,instanceTypeId,instanceFormatIds,instanceFormats,physicalDescriptions,languages,notes,administrativeNotes,modeOfIssuanceId,catalogedDate,previouslyHeld,staffSuppress,discoverySuppress,deleted,statisticalCodeIds,statusId,statusUpdatedDate,tags_instance,metadata_instance,holdingsRecords2,natureOfContentTermIds,username,id_user,barcode,active,type,patronGroup,departments,proxyFor,personal,expirationDate,createdDate,updatedDate,metadata_user,customFields,preferredEmailCommunication,externalSystemId,tags_user,enrollmentDate
0,1f3716fd-50cd-44aa-a992-eba5a6bb6b6a,fff6cbc4-89d8-4b11-912e-899988eef596,1,in00000425095,MARC,12 things to know about political parties / Vi...,12 things to know about political parties /,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[],"[{'value': '23766617', 'identifierTypeId': '7e...","[{'name': 'Hayes, Vicki C.', 'contributorTypeI...",[{'value': 'Political parties--United States--...,"[{'classificationNumber': 'JK2261 .H337 2025',...","[{'publisher': 'Black Rabbit Books', 'place': ...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[48 pages : illustrations ; 24 cm.],[eng],[{'instanceNoteTypeId': '86b6e817-e1bc-42fb-ba...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-20,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-20T18:12:50.827+0000,{'tagList': []},{'createdDate': '2026-07-20T18:12:50.826+00:00...,[],[],cmcclure,1f3716fd-50cd-44aa-a992-eba5a6bb6b6a,60606,True,staff,3ddd748f-5ba1-46a6-87b2-1a45a7243624,[],[],"{'pronouns': 'she/her', 'lastName': 'McClure',...",2028-02-08T04:59:59.999+00:00,2026-06-11T13:25:14.293+00:00,2026-06-11T13:25:14.293+00:00,{'createdDate': '2026-02-03T20:51:19.332+00:00...,{'activePatronGroups': []},[],cmcclure@ebsco.com,NaN,NaN
1,91d619a5-66b5-45f3-8680-42525ec90bcd,4852f533-06d9-413e-b7c3-f97586631a4f,1,in00000425128,MARC,Writing the U.S. Constitution : framework of a...,Writing the U.S. Constitution : framework of a...,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[{'value': 'Perspectives library.'}],"[{'value': 'in00024787139', 'identifierTypeId'...","[{'name': 'Baxter, Roberta, 1952-', 'contribut...",[{'value': 'United States. Constitution--Juven...,"[{'classificationNumber': '342.73029', 'classi...","[{'publisher': 'Cherry Lake Press', 'place': '...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[32 pages: illustrations (some color) ; 24 cm.],[eng],[{'instanceNoteTypeId': '6a2533a7-4de2-4e64-84...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-21,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-21T13:51:44.819+0000,{'tagList': []},{'createdDate': '2026-07-21T13:51:44.818+00:00...,[],[],CHoFOLIOadmin,91d619a5-66b5-45f3-8680-42525ec90bcd,NaN,True,staff,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,[],[],"{'lastName': 'Admin', 'firstName': 'CHoFOLIO',...",2032-12-01T04:59:59.000+00:00,2026-08-02T17:31:36.437+00:00,2026-08-02T17:31:36.437+00:00,{'createdDate': '2024-11-21T15:31:03.109+00:00...,{'activePatronGroups': []},[],NaN,NaN,NaN
2,b898c62b-4080-45d5-aae7-63b8c52a6de3,d7b6c2a9-95ed-4856-91a2-647c8298e97a,1,in00000425230,MARC,Bicycling the Colorado Rockies / Vici De Haan.,Bicycling the Colorado Rockies /,[],"[Rev. & expanded, 2nd rd.]",[],"[{'value': '4749869', 'identifierTypeId': '7e5...","[{'name': 'DeHaan, Vici', 'contributorNameType...",[{'value': 'Bicycle touring--Rocky Mountains--...,[{'classificationNumber': 'GV1045.5.R6 D4 1985...,"[{'publisher': 'Pruett Pub. Co.', 'place': 'Bo...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[114 p

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [34]:
print("Original Instance rows:", len(instances_df))
print("After merging with staff:  ", len(instances_users))


Original Instance rows: 108
After merging with staff:   108


## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


In [37]:
summary =instances_users.groupby('username')['id_instance'].count().sort_values(ascending=False)
print("Summary of instance records created by each staff user from", search_date, "to today:")
print(summary)

Summary of instance records created by each staff user from 2026-05-15 to today:
username
fs00000002       35
CHoFOLIOadmin    32
folio            18
gpadilla         12
cmcclure          9
kmclaughlin       1
Name: id_instance, dtype: int64
